# Dependencies and Stage Boundaries: The Scheduler Algorithm

**Module**: 04_execution_engine  
**Notebook**: 07_dependencies_and_stage_boundaries.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The exact algorithm DAGScheduler uses to traverse dependencies backward and split stages.
- **Why does this topic exist?**: Deep mastery of stage boundary creation enables precise execution prediction.
- **What problem does it solve?**: Unexpected stage creation slowing queries.
- **Where is it used in real-world Data Engineering?**: Senior Data Engineering technical interviews.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Narrow/wide transformations, DAGScheduler |
| **Required depth** | Deep internals: can derive stage boundaries from dependencies |
| **Data Engineer** | Explain how dependencies determine stages and why a job recomputes. |
| **Senior Data Engineer** | Diagnose stage boundaries and choose checkpoint/caching points. |
| **Spark specialist** | Reason about dependency objects, preferred locations, and stage submission. |
| **Next topic** | `05_partitions_and_shuffle/01_partitioning_fundamentals.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Stage boundaries are not arbitrary; they are determined by dependencies. A narrow dependency keeps the
pipeline together, while a shuffle dependency forces a materialization point. Understanding this lets you predict
the DAG, place caches/checkpoints, and explain why losing a shuffle map output can recompute a whole stage.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Dependency** | Relationship between parent/child partitions | Narrow or wide |
| **ShuffleDependency** | Wide dependency | Requires a shuffle and stage boundary |
| **Stage boundary** | Materialization point | Ends one stage, starts another |
| **Preferred location** | Where data is | Used for task locality |
| **Lineage** | How to rebuild | Dependency graph |
| **Checkpoint** | Cut lineage | Durable materialization |

### How it works, step by step

1. The scheduler starts from the final RDD/plan.
2. It recursively visits dependencies.
3. A narrow dependency stays in the current stage.
4. A shuffle dependency creates a new stage for the parent.
5. Parent stages are submitted before the child.
6. Lost shuffle outputs may force a parent stage recomputation.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. The Stage Boundary Algorithm: Traverse backward from terminal RDD, split at ShuffleDependency.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Dependencies_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rdd = spark.sparkContext.parallelize(range(100), 2).map(lambda x: (x%5, x)).reduceByKey(lambda a, b: a+b)
print("Lineage tree showing ShuffledRDD:")
print(rdd.toDebugString().decode('utf-8'))

---

## Experiment: Verifying Stage Split Algorithm

- **1. What are we testing?**: Check stage count in Spark UI.
- **2. What do we expect?**: 2 stages.

In [ ]:
print('Count result:', rdd.count())

**3. What happened?**: Result computed across 2 stages.  
**4. Why did it happen?**: reduceByKey introduced ShuffleDependency.  
**5. Practical Engineering Lesson**: ShuffleDependency is the sole trigger for stage boundary creation.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Placing a checkpoint in a long pipeline

- **Industry**: Machine learning
- **Scenario**: An iterative algorithm has a long lineage and fails after many iterations.
- **Problem**: Recomputing the entire lineage after a failure is expensive and can overflow the driver.
- **Why this concept is relevant**: Checkpointing at a stable intermediate point cuts the lineage and makes recovery cheaper.
- **Implementation idea**: Write the intermediate dataset to reliable storage, read it back, and continue from the checkpoint.
- **Expected result**: Faster recovery and bounded lineage growth.
- **Alternative**: A feature store or a persisted table.
- **Trade-offs**: Checkpointing is simple within a job; a feature store supports cross-job reuse.

### Use Case 2: Explaining stage recomputation

- **Industry**: Logistics
- **Scenario**: A shuffle stage fails after a worker restart and the job recomputes an earlier stage.
- **Problem**: The team needs to know why work was repeated and whether the output is still correct.
- **Why this concept is relevant**: The lost shuffle map outputs belong to a parent stage, so the scheduler must rebuild them from dependencies.
- **Implementation idea**: Inspect stage attempts and shuffle map output status; keep transformations deterministic.
- **Expected result**: A correct result with a known recomputation cost.
- **Alternative**: Replicated shuffle storage.
- **Trade-offs**: Lineage recovery is automatic; replication reduces recomputation but adds storage cost.

---

## Business Value

- Better failure diagnosis and recovery planning.
- More deliberate cache/checkpoint placement.
- Clearer performance reasoning about stages.
- Improved reliability for long-running jobs.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Keep transformations deterministic for correct recomputation.
- Checkpoint long lineages and iterative jobs.
- Monitor stage attempts and lost shuffle outputs.
- Place caches at reuse points, not everywhere.
- Understand that a stage boundary is a materialization point.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Derive stages from a dependency graph

**Steps**

1. Build a pipeline with two shuffles.
2. Draw the dependency graph and mark the shuffle dependencies.
3. Predict the stages, then compare with the UI.

**What to observe**: Each shuffle dependency creates a boundary; narrow dependencies stay in the same stage.

**Healthy result**: Your drawing matches the stage list.

**Problematic result**: A hidden shuffle dependency creates an extra stage.

**Engineering takeaway**: Dependencies are the source of truth for stage boundaries.

---

## Trade-offs and Alternatives

### Advantages

- Predictable stage model
- Automatic recovery
- Clear optimization points

### Limitations

- Recomputation can be expensive
- Dependency graphs can be complex
- Checkpointing adds I/O

### When to use it

- Debugging recomputation
- Checkpoint design
- Stage prediction

### When not to use it

- Non-deterministic transformations
- Checkpointing every step
- Ignoring stage retries

### Alternatives inside Spark

- Checkpointing
- Caching
- DAGScheduler

### Alternatives outside Spark

- Replicated storage
- Workflow orchestrators with retries

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Dependencies
**Problem Statement**: Inspect dependencies array.  
**Input / Context**: rdd.dependencies

In [ ]:
# B

### Exercise 2 (Intermediate): Count Shuffles
**Problem Statement**: Count shuffles in complex pipeline.  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): Verify in UI
**Problem Statement**: Check UI stages graph.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Production Debugging Exercise

### Unexpected recomputation

**Scenario**: A job recomputes a large parent stage after a small task failure.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the failed task lost shuffle map outputs.
2. Inspect stage attempts and map output status.
3. Consider checkpointing or a more reliable shuffle service.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Dependencies):
```python
print([type(d).__name__ for d in rdd.dependencies])
```

#### Solution 2 (Count Shuffles):
```python
print('Count ShuffleDependencies')
```

#### Solution 3 (Verify in UI):
```python
print('Inspected stage DAG in localhost:4040')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Algorithm**: Backward traversal splitting at ShuffleDependency.
- **Stages**: ShuffleMapStage followed by downstream stage.

### Essential Technical Terms
- `ShuffleDependency`: Triggers stage boundary.
- `OneToOneDependency`: Continues stage.

### Common Engineering Mistakes
- **Thinking each transformation is a stage**: Only wide dependencies create stages.

### Interview Questions on This Topic
- **Q: How does DAGScheduler decide stage boundaries?**
  - *A*: By traversing the RDD lineage backwards from the action and cutting a new stage boundary at every ShuffleDependency.


### More Interview Questions

- **Q: How are stages determined?**
  - *A*: By shuffle dependencies: a narrow dependency stays within a stage; a wide/shuffle dependency creates a new stage boundary.
- **Q: Why can a small task failure cause a large recomputation?**
  - *A*: If the failed task was a shuffle map task, its map outputs may be lost, forcing the parent stage to be recomputed.

### What to Study Next
Module 04 complete! Proceed to Module 05: Partitions and Shuffle Internals.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')